# C4: spaCy vektori reči + Ward, provera sa rubrikama (BBC RSS vesti, 2023)
**Tip:** vežba iz tvoje pripreme (PDF `spacy_vektori_ward`). Kratki tekstovi, pa provera klastera sa stvarnom rubrikom.

### Zadatak
U fajlu `bbc_news.csv` su vesti iz BBC RSS kanala (2022 do 2024). Radi se samo sa vestima iz 2023, uzorak od 1500.
- Napraviti `news_text` od `title` i `description`. Prokomentarisati šta kratki tekstovi znače za vektor dobijen usrednjavanjem vektora reči.
- Vektori reči iz `en_core_web_md`, transformacija kao klasa (`BaseEstimator`, `TransformerMixin`). Objasniti zašto je ovde, za razliku od SBERT-a, korisno izbaciti stop reči.
- Izbaciti vesti sa nula-vektorom (objasniti zašto smetaju kosinusnoj sličnosti).
- Ward, dendrogram, silueta za 2 do 8 klastera (bar plot), obrazložiti izbor, veličine klastera.
- Top 10 reči / izraza (unigrami i bigrami), opis klastera uz 10 nasumičnih naslova.
- Tek na kraju uporediti klastere sa kolonom `rubrika` kroz `pd.crosstab`.
- Dodatno: ponoviti sa `method='average'` i uporediti.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist

import matplotlib.pyplot as plt
import seaborn as sb
from pypalettes import load_cmap

from pathlib import Path
from IPython.display import display

import spacy
from sklearn.base import BaseEstimator, TransformerMixin

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'
MODELS_DIR = Path.cwd().parent / 'models'

### Šta je urađeno
- `pubDate` pretvoren u datum, prikazan broj vesti po godinama, zadržana samo 2023.
- Izbačeni duplikati po naslovu (ista vest se ponavlja u RSS-u) i stavke koje nisu tekstualne vesti (link mora da vodi na `/news/` ili `/sport/`).
- Uzet uzorak od 1500 vesti, napravljena `news_text` i pogledana dužina (vesti su kratke, oko 30 reči, pa skraćivanje ne treba).
- Kolona `rubrika` je napravljena iz linka, ali se **ne koristi do poslednjeg koraka**.

In [ ]:
data = pd.read_csv(DATA_DIR / 'bbc_news.csv')
data.info()

In [ ]:
data['pubDate'] = pd.to_datetime(data['pubDate'], format='%a, %d %b %Y %H:%M:%S %Z')
print(data['pubDate'].dt.year.value_counts().sort_index())
data = data[data['pubDate'].dt.year == 2023]

print('Duplikati po naslovu:', data.duplicated(subset='title').sum())
data = data.drop_duplicates(subset='title')

delovi_linka = data['link'].str.extract(r'bbc\.co(?:\.uk|m)/(news|sport)/([a-z]+)')
data = data[delovi_linka[0].notna()]
print(data.shape)

In [ ]:
data = data.sample(n=1500, random_state=RAND_STATE).reset_index(drop=True)

# rubrika: koristi se tek u poslednjem koraku za proveru klastera
delovi_linka = data['link'].str.extract(r'bbc\.co(?:\.uk|m)/(news|sport)/([a-z]+)')
data['rubrika'] = np.where(delovi_linka[0] == 'sport', 'sport', delovi_linka[1])
GLAVNE_RUBRIKE = ['uk', 'world', 'sport', 'business', 'entertainment']
data['rubrika'] = data['rubrika'].where(data['rubrika'].isin(GLAVNE_RUBRIKE), 'ostalo')

data['news_text'] = data['title'] + '. ' + data['description']
print(data['news_text'].str.split().str.len().describe())

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. `spacy.load('en_core_web_md', disable=['parser','ner'])`, klasa `NewsVectorizer(BaseEstimator, TransformerMixin)` sa čišćenjem i prosekom vektora
2. Provera nula-vektora: `np.linalg.norm(embeddings, axis=1) == 0`, izbaci te redove i iz `data`
3. Linkage (`pdist`, Ward), dendrogram, silueta za 2 do 8 (bar plot), izbor i veličine
4. Ključne reči `ngram_range=(1,2)`, grafik, 10 naslova po klasteru (`random_state=1`), imena klastera
5. `pd.crosstab(data['rubrika'], data['clust'])` i komentar
6. Dodatno: `method='average'` i poređenje

---
## Dati kod
Ovo dobijaš uz zadatak.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): prikaz kljucnih reci po klasterima
# Funkciju get_cluster_keywords(cluster_id) pises sam; treba da vrati listu parova (rec, skor).
# Raskomentarisi kad stignes do ovog koraka.

# fig, ax_grid = plt.subplots(nrows=2, ncols=3, figsize=(16, 10), constrained_layout=True)
# axes = ax_grid.flatten()
# cmap = load_cmap("excel_Median")
#
# n_clust = data['clust'].nunique()
# for i in range(n_clust):
#     scored_kwds = get_cluster_keywords(i+1)
#     scored_kwds = sorted(scored_kwds, key=lambda item: item[1])
#     scores = [s for kw, s in scored_kwds]
#     keywords = [kw for kw, s in scored_kwds]
#     axes[i].barh(keywords, scores, color=cmap(i))
#     axes[i].set_title(f"Cluster {i+1}", color=cmap(i))
#     axes[i].grid(visible=True, axis='y', color='gray', alpha=0.35)
#
# for ax in axes[n_clust:]:   # brise prazna polja ako je mreza veca od broja klastera
#     ax.remove()
#
# fig.suptitle("Kljucne reci po klasterima")
# plt.show()

In [ ]:
# DATI KOD (primer): skraceni dendrogram, ako je pun nepregledan
# plt.figure(figsize=(12, 6))
# sch.dendrogram(ward_linkage, truncate_mode='lastp', p=30)
# plt.title('Dendrogram (Ward)')
# plt.ylabel('Distanca')
# plt.show()